# 00 数据准备

本 notebook 完成所有数据清洗，并把中间数据导出到 `output/`，供 01–07 各小题使用。

**处理内容：**
1. 解析 `full_music_data` 的 `artists_id`（数组格式 `[178301]`）
2. 拆出「歌曲—艺术家」关联表
3. 以 `data_by_artist` 为锚，从 `influence_data` 补入 `genre` 和 `active_start`
4. 清洗影响边表

运行一次即可，后续 notebook 直接读 `output/`。

In [1]:
import pandas as pd
import numpy as np
import re, os

DATA = '../data'
OUT = 'output'
os.makedirs(OUT, exist_ok=True)

by_artist = pd.read_csv(f'{DATA}/data_by_artist.csv')
by_year   = pd.read_csv(f'{DATA}/data_by_year.csv')
full      = pd.read_csv(f'{DATA}/full_music_data.csv')
influence = pd.read_csv(f'{DATA}/influence_data.csv')

print('by_artist :', by_artist.shape)
print('by_year   :', by_year.shape)
print('full      :', full.shape)
print('influence :', influence.shape)

by_artist : (5854, 16)
by_year   : (100, 14)
full      : (98340, 19)
influence : (42770, 8)


## 1. 解析 `artists_id` 数组，拆出歌曲—艺术家关联表

In [2]:
def clean_ids(raw):
    """把 '[178301]' 或 '[123, 456]' 解析成 id 列表（字符串）"""
    s = str(raw)
    ids = [i for i in re.split(r'[\[\]\s,]+', s) if i]
    return ids

full['artists_id_list'] = full['artists_id'].apply(clean_ids)
full['n_artists'] = full['artists_id_list'].apply(len)

# 歌曲—艺术家关联表（一首歌 → 多位艺术家）
song_artist = (full[['artists_id_list']]
               .explode('artists_id_list')
               .rename(columns={'artists_id_list': 'artist_id'})
               .reset_index()
               .rename(columns={'index': 'song_index'}))

print('多艺术家合作的歌曲数:', (full['n_artists'] > 1).sum())
print('关联表 song_artist :', song_artist.shape)
print('去重后参与的 (song, artist) 对:', song_artist[['song_index','artist_id']].drop_duplicates().shape)

多艺术家合作的歌曲数: 4410
关联表 song_artist : (103487, 2)
去重后参与的 (song, artist) 对: (103487, 2)


## 2. 从 `influence_data` 提取艺术家的流派与活跃年代

In [3]:
def meta_frame(id_col, genre_col, start_col):
    m = influence[[id_col, genre_col, start_col]].drop_duplicates()
    m = m[m[id_col].notna()]
    m = m.rename(columns={id_col: 'artist_id', genre_col: 'genre', start_col: 'active_start'})
    return m

meta_inf = meta_frame('influencer_id', 'influencer_main_genre', 'influencer_active_start')
meta_fol = meta_frame('follower_id', 'follower_main_genre', 'follower_active_start')

meta = pd.concat([meta_inf, meta_fol]).drop_duplicates('artist_id')
meta['artist_id'] = meta['artist_id'].astype(str)
meta['genre'] = meta['genre'].astype(str).str.replace(';', '', regex=False).str.strip()
meta['genre'] = meta['genre'].replace({'nan': 'Unknown', '': 'Unknown'})

print('influence 中出现的去重艺术家:', meta.shape[0])
print(meta['genre'].value_counts().head(10))

influence 中出现的去重艺术家: 5603
genre
Pop/Rock      2808
R&B            677
Jazz           406
Country        403
Latin          229
Electronic     208
Vocal          162
Reggae         141
Blues          101
Folk            95
Name: count, dtype: int64


## 3. 建立艺术家主表（`data_by_artist` 为锚，补入流派/年代）

In [4]:
by_artist['artist_id'] = by_artist['artist_id'].astype(str)
artists = by_artist.merge(meta, on='artist_id', how='left')

# 没有流派标签的艺术家补 'Unknown'
artists['genre'] = artists['genre'].fillna('Unknown')

print('艺术家主表:', artists.shape)
print('有流派标签的艺术家:', (artists['genre'] != 'Unknown').sum())
print('有活跃年代的艺术家:', artists['active_start'].notna().sum())
print('\n流派分布 top10:')
print(artists['genre'].value_counts().head(10))

艺术家主表: (5854, 18)
有流派标签的艺术家: 5599
有活跃年代的艺术家: 5602

流派分布 top10:
genre
Pop/Rock      2807
R&B            677
Jazz           406
Country        403
Unknown        255
Latin          229
Electronic     208
Vocal          162
Reggae         141
Blues          101
Name: count, dtype: int64


## 4. 清洗影响边表并导出

In [5]:
edges = influence.copy()
for c in ['influencer_id', 'follower_id']:
    edges[c] = edges[c].astype(str)
for g in ['influencer_main_genre', 'follower_main_genre']:
    edges[g] = edges[g].astype(str).str.replace(';', '', regex=False).str.strip()

# 保存 songs 时把 id 列表转回字符串，避免后续解析
full_out = full.drop(columns=['artists_id_list'])
full_out['artists_id'] = full_out['artists_id'].astype(str)

artists.to_csv(f'{OUT}/artists.csv', index=False)
edges.to_csv(f'{OUT}/edges.csv', index=False)
full_out.to_csv(f'{OUT}/songs.csv', index=False)
song_artist.to_csv(f'{OUT}/song_artist.csv', index=False)
by_year.to_csv(f'{OUT}/by_year.csv', index=False)

print('已导出到', OUT, ':')
for f in ['artists.csv','edges.csv','songs.csv','song_artist.csv','by_year.csv']:
    print('  ', f, os.path.getsize(f'{OUT}/{f}'), 'bytes')

已导出到 output :
   artists.csv 853747 bytes
   edges.csv 2842480 bytes
   songs.csv 14009935 bytes
   song_artist.csv 1425309 bytes
   by_year.csv 13385 bytes


## 5. 验证：关键交集核对

In [6]:
inf_ids = set(edges['influencer_id']) | set(edges['follower_id'])
art_ids = set(artists['artist_id'])
mus_ids = set(song_artist['artist_id'])

print('influence 艺术家去重:', len(inf_ids))
print('artists 主表艺术家 :', len(art_ids))
print('songs 涉及艺术家   :', len(mus_ids))
print('influence ∩ artists:', len(inf_ids & art_ids))
print('influence ∩ songs  :', len(inf_ids & mus_ids))
print('artists ∩ songs    :', len(art_ids & mus_ids))

influence 艺术家去重: 5603
artists 主表艺术家 : 5854
songs 涉及艺术家   : 5810
influence ∩ artists: 5602
influence ∩ songs  : 5560
artists ∩ songs    : 5810
